In [1]:
import pandas as pd
import numpy as np

matches = pd.read_csv('../data/processed/matches_clean.csv')
deliveries = pd.read_csv('../data/processed/deliveries_clean.csv')
print("Matches:", matches.shape)
print("Deliveries:", deliveries.shape)

Matches: (1090, 21)
Deliveries: (260920, 17)


In [2]:
def get_win_rate(matches):
    win_rates = {}
    for idx, row in matches.iterrows():
        t1, t2 = row['team1'], row['team2']
        season = row['season']
        past = matches[
            (matches['season'] < season) & (
                ((matches['team1'] == t1) & (matches['team2'] == t2)) |
                ((matches['team1'] == t2) & (matches['team2'] == t1))
            )
        ]
        if len(past) == 0:
            win_rates[idx] = 0.5
        else:
            t1_wins = past[past['winner'] == t1].shape[0]
            win_rates[idx] = round(t1_wins / len(past), 3)
    return win_rates

print("Calculating head-to-head win rates... (takes 1-2 mins, wait for Done!)")
matches['team1_win_rate_vs_team2'] = get_win_rate(matches)
print("Done!")
print(matches['team1_win_rate_vs_team2'].describe())

Calculating head-to-head win rates... (takes 1-2 mins, wait for Done!)
Done!
count    1090.000000
mean        0.490050
std         0.196828
min         0.000000
25%         0.417000
50%         0.500000
75%         0.571000
max         1.000000
Name: team1_win_rate_vs_team2, dtype: float64


In [3]:
matches['team1_won_toss'] = (matches['toss_winner'] == matches['team1']).astype(int)
matches['toss_decision_bat'] = (matches['toss_decision'] == 'bat').astype(int)

print(matches[['toss_winner','team1','team1_won_toss',
               'toss_decision','toss_decision_bat']].head(5))

                   toss_winner                        team1  team1_won_toss  \
0  Royal Challengers Bengaluru  Royal Challengers Bengaluru               1   
1          Chennai Super Kings                 Punjab Kings               0   
2             Rajasthan Royals               Delhi Capitals               0   
3               Mumbai Indians               Mumbai Indians               1   
4              Deccan Chargers        Kolkata Knight Riders               0   

  toss_decision  toss_decision_bat  
0         field                  0  
1           bat                  1  
2           bat                  1  
3           bat                  1  
4           bat                  1  


In [4]:
def get_venue_win_rate(matches):
    venue_rates = {}
    for idx, row in matches.iterrows():
        t1 = row['team1']
        venue = row['venue']
        season = row['season']
        past = matches[
            (matches['season'] < season) &
            (matches['venue'] == venue) &
            ((matches['team1'] == t1) | (matches['team2'] == t1))
        ]
        if len(past) == 0:
            venue_rates[idx] = 0.5
        else:
            wins = past[past['winner'] == t1].shape[0]
            venue_rates[idx] = round(wins / len(past), 3)
    return venue_rates

print("Calculating venue win rates... (takes 1-2 mins, wait for Done!)")
matches['team1_venue_win_rate'] = get_venue_win_rate(matches)
print("Done!")
print(matches['team1_venue_win_rate'].describe())

Calculating venue win rates... (takes 1-2 mins, wait for Done!)
Done!
count    1090.000000
mean        0.511257
std         0.213014
min         0.000000
25%         0.500000
50%         0.500000
75%         0.617000
max         1.000000
Name: team1_venue_win_rate, dtype: float64


In [5]:
def get_recent_form(matches, n=5):
    matches = matches.sort_values(['season','date']).reset_index(drop=True)
    form = []
    for idx, row in matches.iterrows():
        t1 = row['team1']
        past = matches.iloc[:idx]
        past_t1 = past[
            (past['team1'] == t1) | (past['team2'] == t1)
        ].tail(n)
        if len(past_t1) == 0:
            form.append(0.5)
        else:
            wins = past_t1[past_t1['winner'] == t1].shape[0]
            form.append(round(wins / len(past_t1), 3))
    return form

print("Calculating recent form... (takes 1-2 mins, wait for Done!)")
matches['team1_recent_form'] = get_recent_form(matches)
print("Done!")
print(matches['team1_recent_form'].describe())

Calculating recent form... (takes 1-2 mins, wait for Done!)
Done!
count    1090.000000
mean        0.500565
std         0.224589
min         0.000000
25%         0.400000
50%         0.600000
75%         0.600000
max         1.000000
Name: team1_recent_form, dtype: float64


In [6]:
matches['team1_won'] = (matches['winner'] == matches['team1']).astype(int)

features = matches[[
    'team1_win_rate_vs_team2',
    'team1_won_toss',
    'toss_decision_bat',
    'team1_venue_win_rate',
    'team1_recent_form',
    'team1_won'
]].copy()

print("Feature set shape:", features.shape)
print("\nSample rows:")
print(features.head(10))
print("\nTarget split:")
print(features['team1_won'].value_counts())

Feature set shape: (1090, 6)

Sample rows:
   team1_win_rate_vs_team2  team1_won_toss  toss_decision_bat  \
0                      0.5               1                  0   
1                      0.5               0                  1   
2                      0.5               0                  1   
3                      0.5               1                  1   
4                      0.5               0                  1   
5                      0.5               0                  1   
6                      0.5               1                  1   
7                      0.5               0                  0   
8                      0.5               0                  0   
9                      0.5               0                  0   

   team1_venue_win_rate  team1_recent_form  team1_won  
0                   0.5                0.5          0  
1                   0.5                0.5          0  
2                   0.5                0.5          1  
3                

In [7]:
features.to_csv('../data/processed/features.csv', index=False)
print("Saved! Check data/processed/features.csv")

Saved! Check data/processed/features.csv
